<a href="https://colab.research.google.com/github/kannisharath/INFO-5731/blob/main/zeroshot_llama2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To login, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Token: 
Add token as git credential? (Y/n) n
Token is valid (permission: read).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_id = "meta-llama/Llama-2-7b-chat-hf"

In [ ]:
import pandas as pd
from transformers import pipeline

# Load the Excel file
file_path = '/content/sample_data/ZEROSHOT2SAMPLES.xlsx'
df = pd.read_excel(file_path)

# Define the summarization prompt
prompt = """
Given the following document, your task is to generate a summary based on the provided text. Use the content of the document to create a concise summary that captures the main points and key information.
Document:
X: {}
"""

# Function to generate summary
def generate_summary(document_text):
    formatted_prompt = prompt.format(document_text)
    summarization_pipeline = pipeline("text-generation", model="meta-llama/Llama-2-7b-chat-hf")
    result = summarization_pipeline(formatted_prompt)
    return result[0]['generated_text']

# Loop through each row in the DataFrame
for index, row in df.iterrows():
    data = {
        "patent_id": row['patent_id'],

        "abstract": row['abstract'],
        "claims": row['claims'],

    }

    # Prepare the document text from the structured data
    document_text =  data["claims"] + "\n"

    # Generate summary for each document
    summary = generate_summary(document_text)
    print(f"Generated Summary for Patent ID {data['patent_id']}:")
    print(summary)
    print("-" * 80)  # Adding a separator for readability between summaries


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Generated Summary for Patent ID US8701017:

Given the following document, your task is to generate a summary based on the provided text. Use the content of the document to create a concise summary that captures the main points and key information.
Document:
X: We claim: 1. A communications device associated with a presentity comprising:a presence user client communicatively coupled to a presence system via a communication network to receive presence information associated with said presentity said presence information including a respective presentity presence state indicating a respective availability of said presentity as provided by said presence system to each contact on a contact list of said presentity in which at least one said contact is a watcher of said presentity; anda display coupled to said presence user client for displaying to said presentity said contact list of said presentity and adjacent to each said contact in said contact list a respective representation of said re

In [ ]:
pip install pandas torch bert_score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 672.5 kB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
  Using cached nvidia_nccl_cu12-2.19.3-py3-none-manylinux1_x86_64.whl (166.0 MB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-many

In [ ]:
import pandas as pd
import torch
import re
from bert_score import score

# Load the DataFrame from the Excel file
# Load the DataFrame from the CSV file
input_file = "/content/sample_data/patent_data (9).xlsx"  # Modify the path as needed
df = pd.read_excel(input_file)


# Define a list to store the BERT metric scores
bert_metric_scores = []

# Set device to GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Iterate over the rows in the DataFrame
for index, row in df.iterrows():
    # Clean the text to remove non-ASCII characters
    abstract = re.sub(r'[^\x00-\x7F]+', '', str(row['abstract']))
    generated_summary = str(row['Summary'])

    # Calculate BERT metric scores
    _, _, bert_metric_score = score([generated_summary], [abstract], model_type="bert-base-uncased", device=device)

    # Append the score to the BERT metric scores list
    bert_metric_scores.append(bert_metric_score.item())

# Add the scores to the DataFrame
df["BERT_Metric_Score"] = bert_metric_scores

# Save the updated DataFrame to a new Excel file
output_file = "/content/sample_data/zeroshot_bert_score.csv"
df.to_csv(output_file, index=False)

# Print the average BERT metric score
print("Average BERT Metric Score:", sum(bert_metric_scores) / len(bert_metric_scores))

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Average BERT Metric Score: 0.6515281294521532


In [ ]:
pip install pandas rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24933 sha256=6e74c5f1a456acd594a590812133dcc841712a90933d2324871fd245ea192ff8
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge-score


In [ ]:
import pandas as pd
import re
from rouge_score import rouge_scorer

# Load the DataFrame from the Excel file
input_file = "/content/sample_data/patent_data (9).xlsx"  # Modify the path as needed
df = pd.read_excel(input_file)

# Define a list to store the ROUGE scores
rouge_scores = []

# Create a ROUGE scorer instance for ROUGE-1, ROUGE-2, and ROUGE-L
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Iterate over the rows in the DataFrame
for index, row in df.iterrows():
    # Clean the text to remove non-ASCII characters
    abstract = re.sub(r'[^\x00-\x7F]+', '', str(row['abstract']))
    generated_summary = str(row['Summary'])

    # Calculate ROUGE scores
    scores = scorer.score(abstract, generated_summary)

    # Append the scores dictionary to the ROUGE scores list
    rouge_scores.append(scores)

# Add the ROUGE scores to the DataFrame
df['ROUGE_Scores'] = rouge_scores

# Save the updated DataFrame to a new Excel file
output_file = "/content/sample_data/ZEROSHOT_rouge_score.csv"
df.to_csv(output_file, index=False)

# Optionally, print the average scores for each ROUGE metric
average_scores = {metric: 0 for metric in ['rouge1', 'rouge2', 'rougeL']}
for scores in rouge_scores:
    for key in average_scores:
        average_scores[key] += scores[key].fmeasure
for key in average_scores:
    average_scores[key] /= len(rouge_scores)

print("Average ROUGE Scores:", average_scores)

Average ROUGE Scores: {'rouge1': 0.43848046053877193, 'rouge2': 0.25286840596977533, 'rougeL': 0.30126063986660256}


In [ ]:
pip install pandas torch transformers tqdm sklearn numpy

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [ ]:
import pandas as pd
from tqdm import tqdm
import torch
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity
import re
import numpy as np

# Load the DataFrame from the Excel file
input_file = "/content/sample_data/patent_data (9).xlsx"  # Modify the path as needed
df = pd.read_excel(input_file)

# Define a list to store SummaC scores
summac_scores = []

# Initialize BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Set device to GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def get_bert_embeddings(text):
    """Generate BERT embeddings for the given text."""
    inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True).to(device)
    outputs = model(**inputs)
    # Use the pooled output directly to represent the whole sequence
    return outputs['pooler_output'].detach().cpu().numpy()

# Process each row in the DataFrame
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Calculating Scores"):
    abstract = re.sub(r'[^\x00-\x7F]+', ' ', str(row['abstract']))  # Clean non-ASCII characters
    original_text = abstract
    generated_summary = str(row['Summary'])

    # Get embeddings for original text and generated summary
    original_text_embedding = get_bert_embeddings(original_text)
    generated_summary_embedding = get_bert_embeddings(generated_summary)

    # Calculate cosine similarity between embeddings
    cos_sim = cosine_similarity(original_text_embedding, generated_summary_embedding)[0][0]
    summac_scores.append(cos_sim)

# Add the scores to the DataFrame
df['SummaC_Score'] = summac_scores

# Save the updated DataFrame to a new Excel file
output_file = "/content/sample_data/zeroshot_summac_score.csv"
df.to_csv(output_file, index=False)
# Print the average SummaC score
print("Average SummaC Score:", np.mean(summac_scores))

Calculating Scores: 100%|██████████| 19/19 [00:35<00:00,  1.85s/it]

Average SummaC Score: 0.91308856
